# Ensemble Learning
Goals:
- Generate random dataset with 20 features and a binary target, test/compare the following models on class prediction:
  - Logistic Regression
  - KNN
  - SVC
  - Gaussian Naive Bayes
  - Decision Tree
  - Random Forest
  - Extra Trees
- Create six ensemble models and compare predictive performance on the randomly generated data:
  - Hard and soft voting classifiers using Logistic Regression, KNN, SVC , Gaussian Naive Bayes, Decision Tree
  - Hard and soft voting classifiers using Decision Tree, Random Forest, and Extra Trees
  - Hard and soft voting classifiers using Logistic Regression, KNN, and SVC

### Generate random dataset

• n_samples=5000 - *5000 samples*

• n_features=20 - *20 total featues*

• n_informative=5 - *5/20 features correlate with/help determine target class*

• n_redundant=5 - *5/20 features are linear combinations of infromative features (they correlate with target class but don't add new info)*

• n_clusters_per_class=3 - *3 subgroups/clusters within each class*

• weights=[0.60, 0.40] - *60% class 0, 40% class 1*





In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

#generate synthetic data with the given params
X, y = make_classification(
    n_samples=5000,
    n_features=20,
    n_informative=5,
    n_redundant=5,
    n_clusters_per_class=3,
    weights=[0.60, 0.40],
    random_state=20045079
)

#convert features to pandas dataframe
X = pd.DataFrame(X, columns=[f"feature{i}" for i in range(1, 21)])

#train-test split the data (75-25)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y,
                                                    random_state=20045079)

### Base models

The simplest models, Logistic Regression and Naive Bayes, performed the worst with F1 scores of around 0.6.

The medium-complexity nonlinear models, SVC and Decision Tree, performed marginally better with F1 scores around 0.8.

KNN's performance was in between the simplest and medium-complexity models previously mentioned with a F1 of 0.74.

The multi-tree models, Random Forest and Extremely Randomized Trees, performed the best, both achieving a 0.86 F1 score.

The positive correlation between model complexity and performance is due to non-linear feature-target relationships and feature interactions that independence-assuming models fail to capture.

In [ ]:
#standard scaler for preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
scaler = StandardScaler()

#logistic regression
from sklearn.linear_model import LogisticRegression
logreg = LogisticRegression(class_weight='balanced', max_iter=5000, random_state=42, C=0.7)
logreg_pipe = Pipeline([('scaler', scaler), ('logreg', logreg)])

#k-nearest neighbors
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=25, weights='distance')
knn_pipe = Pipeline([('scaler', scaler), ('knn', knn)])

#support vector classifier
from sklearn.svm import SVC
svc = SVC(kernel='rbf', C=3.0, gamma='scale', probability=True, class_weight='balanced',
          random_state=42)
svc_pipe = Pipeline([('scaler', scaler), ('svc', svc)])

#gaussian naive bayes
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()

#decision tree
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(class_weight='balanced', random_state=42, min_samples_leaf=2)

#random forest
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42,
                            n_jobs=-1, min_samples_leaf=2)

#extremely randomized trees
from sklearn.ensemble import ExtraTreesClassifier
et = ExtraTreesClassifier(n_estimators=300, class_weight='balanced', random_state=42,
                          n_jobs=-1, min_samples_leaf=2)

#function that returns model's metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
def evaluate_model(name, model):
    #fit and predict
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    #get metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    conf_matrix = confusion_matrix(y_test, y_pred)
    #return metrics
    return {'accuracy': accuracy, 'precision': precision, 'recall': recall,
            'f1': f1, 'conf_matrix': conf_matrix}

#model dictionary
models = {
    "LogReg": logreg_pipe,
    "KNN": knn_pipe,
    "SVC": svc_pipe,
    "GNB": gnb,
    "Decision Tree": dt,
    "Random Forest": rf,
    "Extra Trees": et
}
model_metrics = {}
#evaluate models
for name, model in models.items():
    model_metrics[name] = evaluate_model(name, model)

#print all metrics
for name, metrics in model_metrics.items():
  print(f"{name} Metrics:")
  for metric, value in metrics.items():
    if isinstance(value, float):
      print(f"{metric}: {round(value, 3)}")
    elif isinstance(value, (int, str)):
      print(f"{metric}: {value}")
    else:
      print(f"{metric}:\n{value}")
  print("\n\n")


LogReg Metrics:
accuracy: 0.653
precision: 0.559
recall: 0.639
f1: 0.597
conf_matrix:
[[495 253]
 [181 321]]



KNN Metrics:
accuracy: 0.819
precision: 0.881
recall: 0.635
f1: 0.738
conf_matrix:
[[705  43]
 [183 319]]



SVC Metrics:
accuracy: 0.863
precision: 0.804
recall: 0.873
f1: 0.837
conf_matrix:
[[641 107]
 [ 64 438]]



GNB Metrics:
accuracy: 0.75
precision: 0.768
recall: 0.54
f1: 0.634
conf_matrix:
[[666  82]
 [231 271]]



Decision Tree Metrics:
accuracy: 0.823
precision: 0.759
recall: 0.821
f1: 0.789
conf_matrix:
[[617 131]
 [ 90 412]]



Random Forest Metrics:
accuracy: 0.893
precision: 0.882
recall: 0.847
f1: 0.864
conf_matrix:
[[691  57]
 [ 77 425]]



Extra Trees Metrics:
accuracy: 0.894
precision: 0.887
recall: 0.843
f1: 0.864
conf_matrix:
[[694  54]
 [ 79 423]]





### Hard & soft voting ensemble models

For **Hard Voting**, Scikit-Learn treates each individual model's binary classification as a "vote" and classifies based on majority vote.

For **Soft Voting**, it averages out each model's predicted probability and assigns the class with the highest probability.


For ensembles A and C, soft voting outperformed hard voting. For ensemble B, hard voting outperformed soft voting.

Soft voting is far more vulnerable to less effective models than hard voting. In hard voting, if one of the models predicts an inaccurate/unusual class probability, it doesn't matter unless it swings the majority vote. In soft voting, an unusual probability can heavily influence the average probability and easily swing a vote.

In [ ]:
from sklearn.ensemble import VotingClassifier

#create ensemble A models
ensembleA_hard = VotingClassifier(estimators=[('LogReg', logreg_pipe), ('KNN', knn_pipe), ('GNB', gnb), ('SVC', svc_pipe), ('DT', dt)], voting='hard')
ensembleA_soft = VotingClassifier(estimators=[('LogReg', logreg_pipe), ('KNN', knn_pipe), ('GNB', gnb), ('SVC', svc_pipe), ('DT', dt)], voting='soft')

#create ensemble B models
ensembleB_hard = VotingClassifier(estimators=[('DT', dt), ('RF', rf), ('ET', et)], voting='hard')
ensembleB_soft = VotingClassifier(estimators=[('DT', dt), ('RF', rf), ('ET', et)], voting='soft')

#create ensemble C
ensembleC_hard = VotingClassifier(estimators=[('LogReg', logreg_pipe), ('SVC', svc_pipe), ('KNN', knn_pipe)], voting='hard')
ensembleC_soft = VotingClassifier(estimators=[('LogReg', logreg_pipe), ('SVC', svc_pipe), ('KNN', knn_pipe)], voting='soft')

#ensemble dictionary
ensembles = {
    "Ensemble A (Hard)": ensembleA_hard,
    "Ensemble A (Soft)": ensembleA_soft,
    "Ensemble B (Hard)": ensembleB_hard,
    "Ensemble B (Soft)": ensembleB_soft,
    "Ensemble C (Hard)": ensembleC_hard,
    "Ensemble C (Soft)": ensembleC_soft
}

ensemble_metrics = {}
#evaluate ensembles
for name, ensemble in ensembles.items():
    ensemble_metrics[name] = evaluate_model(name, ensemble)

#print all metrics
for name, metrics in ensemble_metrics.items():
  print(f"{name} Metrics:")
  for metric, value in metrics.items():
    if isinstance(value, float):
      print(f"{metric}: {round(value, 3)}")
    elif isinstance(value, (int, str)):
      print(f"{metric}: {value}")
    else:
      print(f"{metric}:\n{value}")
  print("\n\n")

Ensemble A (Hard) Metrics:
accuracy: 0.849
precision: 0.883
recall: 0.719
f1: 0.793
conf_matrix:
[[700  48]
 [141 361]]



Ensemble A (Soft) Metrics:
accuracy: 0.865
precision: 0.866
recall: 0.785
f1: 0.823
conf_matrix:
[[687  61]
 [108 394]]



Ensemble B (Hard) Metrics:
accuracy: 0.891
precision: 0.881
recall: 0.843
f1: 0.862
conf_matrix:
[[691  57]
 [ 79 423]]



Ensemble B (Soft) Metrics:
accuracy: 0.862
precision: 0.827
recall: 0.829
f1: 0.828
conf_matrix:
[[661  87]
 [ 86 416]]



Ensemble C (Hard) Metrics:
accuracy: 0.849
precision: 0.863
recall: 0.741
f1: 0.797
conf_matrix:
[[689  59]
 [130 372]]



Ensemble C (Soft) Metrics:
accuracy: 0.865
precision: 0.872
recall: 0.777
f1: 0.822
conf_matrix:
[[691  57]
 [112 390]]





### Final comparison

Extra Trees and Random Forest were the most effective models. They did a great job of diving deep into the granular feature-target relationships and were flexible to any potential dependencies between the features. Ensembling these models with weaker ones via voting will only dilute their effectiveness, hence why they outperformed the ensembles that included them. Perhaps if we ensembled these models with weaker ones via gradient boosting, the model would become even stronger.

In [ ]:
all_metrics_list = []

#append singular models to metrics list
for name, metrics in model_metrics.items():
    all_metrics_list.append({
        'Model': name,
        'Accuracy': metrics['accuracy'],
        'Precision': metrics['precision'],
        'Recall': metrics['recall'],
        'F1': metrics['f1'],
        'Confusion Matrix': metrics['conf_matrix']})

#append ensemble models to metrics list
for name, metrics in ensemble_metrics.items():
    all_metrics_list.append({
        'Model': name,
        'Accuracy': metrics['accuracy'],
        'Precision': metrics['precision'],
        'Recall': metrics['recall'],
        'F1': metrics['f1'],
        'Confusion Matrix': metrics['conf_matrix']})

#convert to df, sort, and display
model_comparison = pd.DataFrame(all_metrics_list)
model_comparison = model_comparison.sort_values(by='F1', ascending=False)
model_comparison

,Model,Accuracy,Precision,Recall,F1,Confusion Matrix
6,Extra Trees,0.8936,0.886792,0.842629,0.864147,"[[694, 54], [79, 423]]"
5,Random Forest,0.8928,0.881743,0.846614,0.863821,"[[691, 57], [77, 425]]"
9,Ensemble B (Hard),0.8912,0.881250,0.842629,0.861507,"[[691, 57], [79, 423]]"
2,SVC,0.8632,0.803670,0.872510,0.836676,"[[641, 107], [64, 438]]"
10,Ensemble B (Soft),0.8616,0.827038,0.828685,0.827861,"[[661, 87], [86, 416]]"
8,Ensemble A (Soft),0.8648,0.865934,0.784861,0.823406,"[[687, 61], [108, 394]]"
12,Ensemble C (Soft),0.8648,0.872483,0.776892,0.821918,"[[691, 57], [112, 390]]"
11,Ensemble C (Hard),0.8488,0.863109,0.741036,0.797428,"[[689, 59], [130, 372]]"
7,Ensemble A (Hard),0.8488,0.882641,0.719124,0.792536,"[[700, 48], [141, 361]]"
4,Decision Tree,0.8232,0.758748,0.820717,0.788517,"[[617, 131], [90, 412]]"
